# RideFlow — Pipeline Driver (parameterized)

One notebook that runs **any domain / system / layer**, chosen by parameters a Task
passes via `EXECUTE NOTEBOOK NB_PIPELINE('domain=marketplace system=rideflow layer=bronze')`
(read from `sys.argv`). Runs the same `runtime/` code as the local deploy — the Snowflake
analogue of a parameterized Databricks/Fabric notebook. Replaces the per-layer notebooks.


In [ ]:
# ── PARAMETERS ──────────────────────────────────────────────────────────────────
# A Task passes these as a single key=value string, e.g.:
#   EXECUTE NOTEBOOK NB_PIPELINE('domain=marketplace system=rideflow layer=bronze')
# read from sys.argv (key=value is immune to arg-position quirks). For an INTERACTIVE run,
# just edit the defaults below (or set e.g. LAYER='silver' in a scratch cell) and re-run.
import sys
_kv = dict(t.split('=', 1) for t in ' '.join(sys.argv).split() if '=' in t)
DOMAIN = _kv.get('domain', 'marketplace')
SYSTEM = _kv.get('system', 'rideflow')
LAYER  = _kv.get('layer',  'all')     # provision | landing | bronze | silver | gold | all
DB     = _kv.get('db',     'RIDEFLOW_DEV_DEMO')
print(f'domain={DOMAIN} system={SYSTEM} layer={LAYER} db={DB}')


In [ ]:
# Install LakeLogic >=1.41.0 (the version with the Snowflake-engine fixes). Prefer PyPI
# (works in interactive notebooks / accounts whose runtime reaches the package index).
# Headless Task runs don't have that index, so fall back to the wheelhouse staged in
# Snowflake. TO DROP THE WHEELHOUSE once PyPI is reachable everywhere for you: delete the
# `else` branch below (and the wheelhouse staging in deploy_project.py / sql/setup.sql).
import os, subprocess, sys
def _pip(*a):
    return subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *a]).returncode
if _pip('lakelogic>=1.41.0') != 0:
    from snowflake.snowpark.context import get_active_session
    _s = get_active_session(); os.makedirs('/tmp/wl', exist_ok=True)
    _s.file.get('@RIDEFLOW_DEV_DEMO.PIPELINE.WHEELS', '/tmp/wl')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index',
                    '--find-links', '/tmp/wl', '--no-deps', 'lakelogic', 'loguru', 'sqlglot'], check=True)


In [ ]:
import os, sys
d = os.getcwd()
while d != os.path.dirname(d) and not os.path.isdir(os.path.join(d, 'domains_rideflow')):
    d = os.path.dirname(d)
REPO = d; sys.path.insert(0, REPO); os.chdir(REPO)
SYS = f'domains_rideflow/{DOMAIN}/{SYSTEM}/_system.yaml'
os.environ['SNOWFLAKE_DATABASE'] = DB

from snowflake.snowpark.context import get_active_session
session = get_active_session()
conn = session.connection
session.sql(f'USE DATABASE {DB}').collect()          # current DB+schema for engine TEMP tables
session.sql(f'USE SCHEMA {DB}.PIPELINE').collect()
from lakelogic.engines.snowflake import SnowflakeAdapter
SnowflakeAdapter.set_shared_connection(conn)


In [ ]:
from runtime.medallion import run_layer
from runtime.landing import load_system_landing
import pandas as pd

def run_one(layer):
    if layer == 'provision':
        for s in ['LANDING', 'QUARANTINE', DOMAIN.upper(), 'PIPELINE']:
            session.sql(f'CREATE SCHEMA IF NOT EXISTS {DB}.{s}').collect()
        return [('schemas', 'ok', 0, 0)]
    if layer == 'landing':
        return [(t, 'ok', n, 0) for t, n in load_system_landing(conn, DB, DOMAIN, SYSTEM, via_stage=True)]
    return run_layer(conn, DB, SYS, layer)

layers = ['provision', 'landing', 'bronze', 'silver', 'gold'] if LAYER == 'all' else [LAYER]
rows = [(L,) + tuple(r) for L in layers for r in run_one(L)]
pd.DataFrame(rows, columns=['layer', 'entity', 'status', 'good', 'quarantined'])
